In [1]:
import os
os.chdir('../')

In [2]:
import pandas as pd
from time import sleep

from textwave.modules.generator.question_answering import QAGeneratorMistral
from textwave.modules.utils.metrics import Matching

print('DONE')

DONE


In [61]:
QUESTIONS_PATH = 'textwave/qa_resources/question.tsv'
CORPUS_PATH = 'textwave/storage/'
CHUNKING_STRATEGY = 'fixed-length' # 'fixed-length' or 'sentence'
CHUNKING_PARAMETERS = {
    "chunk_size": 150, 
    "overlap_size": 0
}
INDEX_STRATEGY = "bruteforce"
INDEX_PARAMETERS = {
    'metric': 'cosine',
}
K_NEAREST_NEIGHBORS = 3
MISTRAL_MODEL = 'mistral-large-latest'
# mistral-small-latest
# mistral-medium-latest
# mistral-large-latest
API_KEY = os.environ["MISTRAL_API_KEY"]

In [62]:
# Due to slow runtime, I need to reduce the number of questions I ask MISTRAL
# Create a dataframe with 20 of each type of question (easy, medium, hard)

# Process questions df
raw_questions = pd.read_table(QUESTIONS_PATH)
easy = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'easy'].reset_index(drop=True)[:20]
medium = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'medium'].reset_index(drop=True)[:20]
hard = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'hard'].reset_index(drop=True)[:20]
questions = easy.append(medium).append(hard).reset_index(drop=True)

# Get unique questions 
unique_questions = questions['Question'].unique()
mistral = QAGeneratorMistral(API_KEY, generator_model=MISTRAL_MODEL)
results = {}
count = 0
for question in unique_questions:
    print(f'{count+1}/{len(unique_questions)}')

    # Trigger QA object to ping MISTRAL, get reponse, return
    answer = mistral.generate_answer(query=question, context=[])
    results[question] = answer

    # Increment count
    count += 1

results

/var/folders/vv/ht5sv34157x8n42p5pms345m0000gn/T/ipykernel_3437/2450983998.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  questions = easy.append(medium).append(hard).reset_index(drop=True)


1/41
2/41
3/41
4/41
5/41
6/41
7/41
8/41
9/41
10/41
11/41
12/41
13/41
14/41
15/41
16/41
17/41
18/41
19/41
20/41
21/41
22/41
23/41
24/41
25/41
26/41
27/41
28/41
29/41
30/41
31/41
32/41
33/41
34/41
35/41
36/41
37/41
38/41
39/41
40/41
41/41


{'Was Abraham Lincoln the sixteenth President of the United States?': 'Yes, Abraham Lincoln was the **sixteenth President of the United States**, serving from **March 4, 1861**, until his assassination on **April 15, 1865**. He led the nation during the **Civil War** and is best known for preserving the Union and issuing the **Emancipation Proclamation**.',
 'Did Lincoln sign the National Banking Act of 1863?': 'Yes, **President Abraham Lincoln** signed the **National Banking Act of 1863** into law on **February 25, 1863**. This legislation established a national banking system, created a uniform currency, and helped finance the Union’s efforts during the Civil War.',
 'Did his mother die of pneumonia?': "I can't determine whether his mother died of pneumonia based on the provided context, as there is no relevant information given. Please clarify or provide more details.",
 "How many long was Lincoln's formal education?": 'Abraham Lincoln received **less than one year** of **formal sch

In [63]:
# Get metrcis, add to dataframe
metrics = Matching()

processed = questions[~questions['Question'].isna()]
processed = processed[~processed['Answer'].isna()]

indices = []
for idx, row in processed.iterrows():
    if row['Question'] not in results:
        indices.append(idx)

processed = processed.drop(indices)

for idx, row in processed.iterrows():
    question = row['Question']
    print(f'{idx}/{len(processed)}')

    true_answer = row['Answer']
    generated_answer = results[question]

    em = metrics.exact_match(generated_answer, true_answer)
    print(f"Exact Match: {em}")

    scores, match = metrics.transformer_match(generated_answer, true_answer, question)
    print(f"Transformer Match: {match} | Scores: {scores}\n")

    processed.at[idx, 'Exact Match'] = em
    processed.at[idx, 'Transformer Match'] = match


Using device: cpu
0/59
Exact Match: True
Transformer Match: True | Scores: {'yes': {'Yes, Abraham Lincoln was the **sixteenth President of the United States**, serving from **March 4, 1861**, until his assassination on **April 15, 1865**. He led the nation during the **Civil War** and is best known for preserving the Union and issuing the **Emancipation Proclamation**.': 1.0}}

1/59
Exact Match: True
Transformer Match: True | Scores: {'Yes.': {'Yes, Abraham Lincoln was the **sixteenth President of the United States**, serving from **March 4, 1861**, until his assassination on **April 15, 1865**. He led the nation during the **Civil War** and is best known for preserving the Union and issuing the **Emancipation Proclamation**.': 1.0}}

2/59
Exact Match: True
Transformer Match: True | Scores: {'Yes.': {'Yes, **President Abraham Lincoln** signed the **National Banking Act of 1863** into law on **February 25, 1863**. This legislation established a national banking system, created a uniform

In [64]:
n = processed[~processed['Exact Match'].isna()]
easy = n[n['DifficultyFromQuestioner'] == 'easy']
medium = n[n['DifficultyFromQuestioner'] == 'medium']
hard = n[n['DifficultyFromQuestioner'] == 'hard']

dfs = [easy, medium, hard]

for df in dfs:
    print(len(df[df['Exact Match']==True]) / len(df))
    print(len(df[df['Transformer Match']==True]) / len(df))
    


1.0
1.0
0.625
0.6875
0.55
0.55
